# UrbanSense AI — YOLO11n Road Damage Detection (Kaggle P100/T4)

This notebook fine-tunes **YOLO11n** on the **RDD2022 India** road damage dataset.

### Target 4-Class Taxonomy:
- **Class 0 (D00)**: Longitudinal Crack (wheel path fatigue)
- **Class 1 (D10)**: Transverse Crack (cross-road thermal/reflective crack)
- **Class 2 (D20)**: Alligator Crack (fatigue mesh cracking)
- **Class 3 (D40)**: Pothole (road surface cavity)

> **Orientation Guard**: Road crack orientation is physically meaningful: longitudinal cracks are parallel to travel, transverse cracks are perpendicular. Thus, rotation augmentations (`degrees=0.0`) and vertical flips (`flipud=0.0`) are strictly disabled to prevent label corruption.

## 1. Hardware & GPU Diagnostic

In [ ]:
import torch
import os

print(f'PyTorch Version: {torch.__version__}')
print(f'CUDA Available:  {torch.cuda.is_available()}')
if torch.cuda.is_available():
    device_name = torch.cuda.get_device_name(0)
    total_vram = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f'GPU Device:      {device_name}')
    print(f'Total VRAM:      {total_vram:.2f} GB')
    if total_vram < 14:
        print('Recommendation: VRAM < 14 GB. Keep batch size at 8.')
    else:
        print('Recommendation: VRAM >= 14 GB. Batch size 8 or 16 supported.')
else:
    print('❌ ERROR: No GPU accelerator detected! In Kaggle menu: Settings -> Accelerator -> GPU P100 / T4.')


## 2. Install Ultralytics & Dependencies

In [ ]:
!pip install -q --upgrade 'ultralytics>=8.3.0' opencv-python matplotlib pyyaml

import ultralytics
ultralytics.checks()


## 3. Dataset Configuration & Portability Setup

Detects whether the dataset is mounted via Kaggle input dataset, unzipped in `/kaggle/working/`, or cloned from the UrbanSense AI repository. Generates a runtime `dataset_kaggle.yaml` pointing to the exact path.

In [ ]:
from pathlib import Path
import yaml

# Search common Kaggle paths for the prepared rdd2022 dataset
potential_roots = [
    Path('/kaggle/input/urbansense-rdd2022/rdd2022'),
    Path('/kaggle/input/rdd2022-india/rdd2022'),
    Path('/kaggle/input/rdd2022-india'),
    Path('/kaggle/working/UrbanSense-AI/datasets/rdd2022'),
    Path('/kaggle/working/datasets/rdd2022'),
    Path('./datasets/rdd2022'),
]

dataset_root = None
for p in potential_roots:
    if (p / 'images' / 'train').is_dir() and (p / 'labels' / 'train').is_dir():
        dataset_root = p.resolve()
        break

if dataset_root is None:
    print('⚠️ Dataset not found in standard paths. If you uploaded a zip, please specify the extraction path.')
    # Default fallback working directory
    dataset_root = Path('/kaggle/working/datasets/rdd2022').resolve()
else:
    print(f'✅ Found prepared RDD2022 dataset at: {dataset_root}')

# Generate runtime dataset YAML for Kaggle
kaggle_yaml_path = Path('/kaggle/working/dataset_kaggle.yaml')
yaml_content = {
    'path': str(dataset_root),
    'train': 'images/train',
    'val': 'images/val',
    'names': {
        0: 'longitudinal_crack',
        1: 'transverse_crack',
        2: 'alligator_crack',
        3: 'pothole',
    },
}
with open(kaggle_yaml_path, 'w') as f:
    yaml.dump(yaml_content, f, sort_keys=False)

print(f'Generated Kaggle dataset specification at: {kaggle_yaml_path}')
print(yaml_content)


## 4. Dataset Audit & Split Verification

In [ ]:
from collections import Counter

train_img_dir = dataset_root / 'images' / 'train'
val_img_dir = dataset_root / 'images' / 'val'
train_lbl_dir = dataset_root / 'labels' / 'train'
val_lbl_dir = dataset_root / 'labels' / 'val'

train_imgs = list(train_img_dir.glob('*.jpg')) if train_img_dir.is_dir() else []
val_imgs = list(val_img_dir.glob('*.jpg')) if val_img_dir.is_dir() else []
print(f'Training images:   {len(train_imgs)}')
print(f'Validation images: {len(val_imgs)}')

def count_labels(lbl_dir):
    counts = Counter()
    empty = 0
    for p in lbl_dir.glob('*.txt'):
        text = p.read_text().strip()
        if not text:
            empty += 1
            continue
        for line in text.splitlines():
            parts = line.strip().split()
            if parts:
                counts[int(parts[0])] += 1
    return counts, empty

if train_lbl_dir.is_dir():
    t_counts, t_empty = count_labels(train_lbl_dir)
    v_counts, v_empty = count_labels(val_lbl_dir)
    print(f'Negative background samples: Train={t_empty}, Val={v_empty}')
    names = ['longitudinal_crack', 'transverse_crack', 'alligator_crack', 'pothole']
    for i, name in enumerate(names):
        print(f'  Class {i} ({name}): Train={t_counts[i]}, Val={v_counts[i]}')


## 5. Initialize Pretrained YOLO11n

Downloads standard YOLO11n weights pretrained on COCO80, then adapts the classification head to the 4 road distress classes.

In [ ]:
from ultralytics import YOLO

model = YOLO('yolo11n.pt')
print('Model architecture initialized: YOLO11n')


## 6. Training Execution (50 Epochs, AdamW, Orientation Guards)

### Hyperparameter Configuration:
- **Epochs**: `50`
- **Batch Size**: `8` (adjust to `16` if VRAM is $\ge 15$ GB)
- **Optimizer**: `AdamW` with initial learning rate `lr0=0.001`
- **Early Stopping**: `patience=10`
- **Workers**: `2`
- **Random Seed**: `42` (reproducibility)
- **Orientation Augmentations**:
  - `degrees=0.0` (STRICT: prevents longitudinal crack D00 from becoming transverse crack D10)
  - `flipud=0.0` (STRICT: prevents inverted road perspective)
  - `fliplr=0.5` (safe horizontal flip)
  - `mosaic=1.0` (effective multi-image context mosaic)

In [ ]:
project_dir = '/kaggle/working/runs/train_road_damage'
run_name = 'yolo11n_rdd2022_india'

results = model.train(
    data=str(kaggle_yaml_path),
    epochs=50,
    patience=10,
    batch=8,
    imgsz=640,
    device=0,
    workers=2,
    seed=42,
    project=project_dir,
    name=run_name,
    optimizer='AdamW',
    lr0=0.001,
    save=True,
    save_period=5,
    val=True,
    plots=True,
    verbose=True,
    # Orientation-safe augmentations
    degrees=0.0,
    fliplr=0.5,
    flipud=0.0,
    mixup=0.0,
    mosaic=1.0,
)


## 7. Resume Checkpoint (Optional / Recovery)

If the Kaggle 9-hour execution limit or network disconnect interrupts training, run this cell to automatically resume from `last.pt`.

In [ ]:
# To resume an interrupted training session, uncomment and execute:
# last_checkpoint = Path(project_dir) / run_name / 'weights' / 'last.pt'
# if last_checkpoint.is_file():
#     print(f'Resuming from checkpoint: {last_checkpoint}')
#     resume_model = YOLO(str(last_checkpoint))
#     resume_model.train(resume=True)
# else:
#     print('No checkpoint found to resume.')


## 8. Validation Evaluation & Per-Class Metrics

In [ ]:
best_weights = Path(project_dir) / run_name / 'weights' / 'best.pt'
if best_weights.is_file():
    print(f'Evaluating best model: {best_weights}')
    eval_model = YOLO(str(best_weights))
    metrics = eval_model.val(data=str(kaggle_yaml_path), split='val')
    print(f'\nOverall mAP50:    {metrics.box.map50:.4f}')
    print(f'Overall mAP50-95: {metrics.box.map:.4f}')
    print(f'Overall Precision: {metrics.box.mp:.4f}')
    print(f'Overall Recall:    {metrics.box.mr:.4f}')
    print('\nPer-Class Performance:')
    names = ['longitudinal_crack', 'transverse_crack', 'alligator_crack', 'pothole']
    for i, name in enumerate(names):
        p = metrics.box.p[i] if i < len(metrics.box.p) else 0.0
        r = metrics.box.r[i] if i < len(metrics.box.r) else 0.0
        ap50 = metrics.box.ap50[i] if i < len(metrics.box.ap50) else 0.0
        print(f'  {name:<20} | Precision: {p:.3f} | Recall: {r:.3f} | mAP50: {ap50:.3f}')
else:
    print(f'Weights not found at {best_weights}. Complete training first.')


## 9. Plot Inspection

In [ ]:
import matplotlib.pyplot as plt
import cv2

run_dir = Path(project_dir) / run_name
plots = ['results.png', 'confusion_matrix.png', 'PR_curve.png', 'F1_curve.png']

for plot_name in plots:
    plot_path = run_dir / plot_name
    if plot_path.is_file():
        img = cv2.imread(str(plot_path))
        if img is not None:
            img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            plt.figure(figsize=(12, 6))
            plt.imshow(img_rgb)
            plt.axis('off')
            plt.title(plot_name)
            plt.show()


## 10. Model Download & UrbanSense AI Integration

### Deployment Target:
Copy the downloaded `best.pt` file to your local UrbanSense AI workspace at:
```
ai_engine/models/road_damage_yolo11n.pt
```

Once placed, `RoadDamageDetector` automatically loads the weights and begins processing road defects without code changes.

In [ ]:
from IPython.display import FileLink

best_weights = Path(project_dir) / run_name / 'weights' / 'best.pt'
if best_weights.is_file():
    print(f'File size: {best_weights.stat().st_size / (1024*1024):.2f} MB')
    display(FileLink(str(best_weights)))
    print('\nClick the link above to download best.pt to your computer.')
    print('Destination: ai_engine/models/road_damage_yolo11n.pt')
else:
    print('best.pt not yet generated.')
